In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
!pip install torch

In [ ]:
import pandas as pd
import os
import glob
from sklearn.model_selection import train_test_split
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

In [ ]:



      #allll of this is only to get 1- the paths and split this data
import pandas as pd
import os
import glob
from sklearn.model_selection import train_test_split

# 1. Define Paths
root_dir = os.path.join(path, "dataset")
img_dir = os.path.join(root_dir, "images")                # u need image dir, and masks dir this time!!
mask_dir = os.path.join(root_dir, "masks")

# 2. Get All Filenames (Sorted to ensure alignment)

image_files = sorted(glob.glob(f"{img_dir}/*.jpg"))
mask_files  = sorted(glob.glob(f"{mask_dir}/*.png"))

# 3. Create DataFrame
# Construct full paths or relative paths
data = []
for img_name in image_files:
        data.append({
            "image_path": os.path.join(img_dir, img_name),
            "mask_path": os.path.join(mask_dir, img_name)
        })

df = pd.DataFrame(data)

# 4. Split Data (80% Train, 20% Test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Total: {len(df)}, Train: {len(train_df)}, Test: {len(test_df)}")

In [ ]:
print(root_dir)
print(img_dir)


In [ ]:
print(image_files)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
class UnderwaterDataset(Dataset):
  def __init__(self, df, transform=None, mask_transform=None):
        self.df = df
        self.transform = transform
        self.mask_transform = mask_transform

  def __len__(self):
        return len(self.df)

  def __getitem__(self, idx):
        # 1. Get Paths from DataFrame
        row = self.df.iloc[idx]
        img_path = row['image_path']
        mask_path = row['mask_path']

        # 2. Load Image & Mask
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") # Grayscale for masks

        # 3. Apply Transforms
        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        # 3. Remap Mask (we need this because CrossEntropy requires the labels to be consecutive)
        mask = remap_mask(mask)

        return image, mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Image transforms (Resize, ToTensor, Normalize with ImageNet stats)
image_transforms = transforms.Compose([
  # TODO: Add transforms
  # Hint: ToTensor, Resize to (256, 256), Normalize with ImageNet mean/std
  # YOUR CODE HERE
  transforms.ToTensor(),
  transforms.Resize((256, 256)),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mask transforms (Resize, PILToTensor)
mask_transforms = transforms.Compose([
  # TODO: Add transforms
  # Hint: Resize to (256, 256) with NEAREST interpolation, PILToTensor
  # YOUR CODE HERE
  transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
  transforms.PILToTensor(),
])

In [ ]:

# Create Dataset objects
train_dataset = UnderwaterDataset(train_df, transform=image_transforms, mask_transform=mask_transforms)
test_dataset = UnderwaterDataset(test_df, transform=image_transforms, mask_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
def denormalize(img):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
  img = img * std + mean
  img = np.clip(img, 0, 1)
  return img

In [ ]:
# Display 4 images with their masks side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # TODO: Get an image-mask pair from train_dataset
    # Hint: Use train_dataset[i] to get the i-th sample

    # YOUR CODE HERE
    image, mask = train_dataset[i]
    print(image.shape)
    # Display image (denormalize first)
    axes[0, i].imshow(denormalize(image),cmap="gray")
    axes[0, i].set_title(f"see Image {i+1}")
    axes[0, i].axis("off")

    # Display mask
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title(f"see Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
# Define U-Net Model
import segmentation_models_pytorch as smp
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=1,  # Binary segmentation (1 output channel)
).to(device)

In [ ]:
# TO DO
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device
    images, masks = images.to(device), masks.to(device).float()

    # TODO: Complete the training step
    # 1. Forward pass
    # 2. Compute loss
    # 3. Zero gradients
    # 4. Backward pass
    # 5. Update weights

    # YOUR CODE HERE
    outputs = model(images)
    loss = criterion(outputs, masks)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)




  # TODO: Task 6 (continued) - Complete the validation function

def validate(model, dataloader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
      images, masks = images.to(device), masks.to(device).float()

      # TODO: Complete the validation step
      # 1. Forward pass
      # 2. Compute loss

      # YOUR CODE HERE
      outputs = model(images)
      loss = criterion(outputs, masks)

      total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
from torch import nn

# YOUR CODE HERE
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 1  # Train for 5 epochs

# Run training
train_losses = []
val_losses = []

for epoch in range(num_epochs):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
  val_loss = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# TO DO
# TODO: Task 8 - Visualize predictions on test images
import random

model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]



  with torch.no_grad():
    input_tensor = image.unsqueeze(0).to(device)
    output = model(input_tensor)


  # Display results
  axes[i, 0].imshow(denormalize(image))
  axes[i, 0].set_title("MRI Image")
  axes[i, 0].axis("off")

  axes[i, 1].imshow(mask.squeeze(), cmap="gray")
  axes[i, 1].set_title("Ground Truth")
  axes[i, 1].axis("off")

  axes[i, 2].imshow(pred.squeeze(), cmap="gray")
  axes[i, 2].set_title("Prediction")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()